# Methods playground — trim & time helpers

Exercises four helpers from `inference.methods`:

1. `time_to_sample_indices`
2. `trim_waveform`
3. `time_to_atom_indices`
4. `trim_atoms_contexts`

Run cells **top to bottom**. Loading `get_inference_engine()` pulls SCAPES checkpoints (slow once).

**Tip:** start Jupyter with cwd under `app/backend`, or run the setup cell from that folder (`cd app/backend` then `jupyter notebook`).

In [1]:
from pathlib import Path
import sys


def find_backend_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "inference" / "methods.py").exists():
            return p
    raise FileNotFoundError(
        "Could not find app/backend (need inference/methods.py on path).\n"
        "cd app/backend before launching Jupyter, or open notebook via Cursor run-from-backend."
    )


BACKEND_ROOT = find_backend_root()
if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0, str(BACKEND_ROOT))

print("BACKEND_ROOT =", BACKEND_ROOT)

BACKEND_ROOT = /Users/claracharbonnier/Documents/REPO/S103-Interface-for-Generative-Audio-Latent-Interpolation/app/backend


In [3]:
import torch

from inference.constants import AUDIO_ASSET_MAP
from inference.models import AudioElement
from inference.methods import (
    get_inference_engine,
    time_to_sample_indices,
    time_to_atom_indices,
    trim_waveform,
    trim_atoms_contexts,
)

In [4]:
engine = get_inference_engine()
SR = engine.sr
HOP = engine.hop_samples
SEG = engine.segment_samples
print(f"sr={SR}, hop_samples={HOP}, segment_samples={SEG}")

Initializing EnCodec processor on device: cpu
Streamable mode: True
  → Enabling streamable mode (disabling chunking)
✓ EnCodec 48kHz model loaded (HuggingFace) - streamable
✓ Sample rate: 48000 Hz
✓ Frame rate: 150 Hz
sr=48000, hop_samples=4800, segment_samples=6720


In [5]:
wav_path = AUDIO_ASSET_MAP[AudioElement.CAMPFIRE]
assert wav_path.exists(), f"Missing asset: {wav_path}"

audio_full = engine.load_audio_to_tensor(str(wav_path))
MAX_SEC = 3
audio = audio_full[:, :, : SR * MAX_SEC]
T = audio.shape[-1]
dur_sec = T / SR
print(wav_path.name, "shape", tuple(audio.shape), "~", round(dur_sec, 3), "s")

camp_fire.wav shape (1, 2, 144000) ~ 3.0 s


## 1–2. `time_to_sample_indices` + `trim_waveform`

In [6]:
START_SEC = 0.5
END_SEC = min(2.0, dur_sec - 0.05)
assert START_SEC < END_SEC

s0, s1 = time_to_sample_indices(START_SEC, END_SEC, sample_rate=SR, num_samples=T)
clip = trim_waveform(audio, s0, s1)

assert clip.shape[-1] == s1 - s0
print(f"[{START_SEC}, {END_SEC}) s -> samples [{s0}, {s1}), waveform_len={clip.shape[-1]}")

# Full clip via None bounds
s_full_a, s_full_b = time_to_sample_indices(None, None, sample_rate=SR, num_samples=T)
assert (s_full_a, s_full_b) == (0, T)
whole = trim_waveform(audio, s_full_a, s_full_b)
assert whole.shape[-1] == T
print("None bounds ->", (s_full_a, s_full_b), "OK")

# Inverted range should raise
try:
    time_to_sample_indices(END_SEC, START_SEC, sample_rate=SR, num_samples=T)
except ValueError as e:
    print("expected ValueError:", e)
else:
    raise AssertionError("expected ValueError for inverted time window")

print("time_to_sample_indices + trim_waveform: PASSED")

[0.5, 2.0) s -> samples [24000, 96000), waveform_len=72000
None bounds -> (0, 144000) OK
expected ValueError: empty or inverted sample range: start_sample=96000, end_sample=24000
time_to_sample_indices + trim_waveform: PASSED


## 3–4. `time_to_atom_indices` + `trim_atoms_contexts`

Loads **`{stem}_atoms.pt`** / **`{stem}_contexts.pt`** next to the WAV (same as `precompute_cache`). Falls back to cwd if needed.

If you still see an old error under this cell, use **Clear outputs** or **Restart kernel** — Jupyter keeps previous run output until you re-execute.

In [ ]:
# Load atoms/contexts from disk (same layout as precompute_cache: beside the WAV).
from pathlib import Path

wav_path = Path(wav_path)
stem = wav_path.stem
beside_atoms = wav_path.parent / f"{stem}_atoms.pt"
beside_ctx = wav_path.parent / f"{stem}_contexts.pt"
cwd_atoms = Path(f"{stem}_atoms.pt")
cwd_ctx = Path(f"{stem}_contexts.pt")

if beside_atoms.is_file() and beside_ctx.is_file():
    atoms_path, contexts_path = beside_atoms, beside_ctx
elif cwd_atoms.is_file() and cwd_ctx.is_file():
    atoms_path, contexts_path = cwd_atoms, cwd_ctx
else:
    raise FileNotFoundError(
        "Missing atom caches.\n"
        f"  Tried next to WAV: {beside_atoms}\n"
        f"  Tried cwd: {cwd_atoms}\n"
        "From app/backend run: python -m inference.precompute_cache"
    )

atoms = torch.load(atoms_path)
contexts = torch.load(contexts_path)

n = len(atoms)
assert len(contexts) == n
print(f"loaded cached atoms/contexts: {n} ← {atoms_path}")

In [ ]:
START_SEC = 0.5
max_end_sec = ((n - 1) * HOP + SEG) / SR
END_SEC = min(2.5, max_end_sec - 0.05)
assert START_SEC < END_SEC

lo, hi = time_to_atom_indices(START_SEC, END_SEC, engine=engine, num_atoms=n)
ta, tc = trim_atoms_contexts(atoms, contexts, START_SEC, END_SEC, engine=engine)

assert hi > lo
assert len(ta) == hi - lo == len(tc)
for i in range(len(ta)):
    assert ta[i] is atoms[lo + i] and tc[i] is contexts[lo + i]
print(f"[{START_SEC}, {END_SEC}) s -> atom indices [{lo}, {hi}), n_trim={hi - lo}")

# Full timeline via None bounds
lo_f, hi_f = time_to_atom_indices(None, None, engine=engine, num_atoms=n)
ta_f, tc_f = trim_atoms_contexts(atoms, contexts, None, None, engine=engine)
assert len(ta_f) == hi_f - lo_f == n
print("None atom bounds ->", (lo_f, hi_f), "length", hi_f - lo_f, "OK")

print("time_to_atom_indices + trim_atoms_contexts: PASSED")

## Sanity check

Waveform trim span (samples) and atom trim span should target comparable timeline regions (exact equality is not required because atoms sit on a hop grid).

In [ ]:
s0, s1 = time_to_sample_indices(START_SEC, END_SEC, sample_rate=SR, num_samples=T)
atom_mid_samples = [k * HOP + SEG // 2 for k in range(lo, hi)]
if atom_mid_samples:
    inside = sum(s0 <= s < s1 for s in atom_mid_samples)
    print(f"midpoints inside waveform slice [{s0},{s1}): {inside}/{len(atom_mid_samples)}")

print("All playground checks finished.")